# 00 — Workspace Setup

<!-- contract -->

| | |
|---|---|
| **Reads** | Nothing |
| **Writes** | Unity Catalog schema + volumes |
| **Runtime** | ~1 min |
| **Requires** | Serverless environment version 5; `flights_sample_3m.csv` uploaded |

Idempotent. Run this first, and re-run it any time you come back to a fresh workspace.

Creates the Unity Catalog namespace, verifies the serverless environment supports Spark ML,
and checks that the source data has been uploaded.

**Manual prerequisites (not automatable, see README):**

1. Databricks Free Edition account with LinkedIn verification completed
2. Serverless **environment version 5** selected in the Environment side panel
3. `flights_sample_3m.csv` uploaded to `/Volumes/workspace/flights/raw/`

This notebook contains no credentials and no personal paths. Anything that would print
credential material belongs in a scratch notebook outside the Git folder.


In [0]:
# Environment probe — env v5 is required for pyspark.ml and mlflow.spark on serverless
import sys

import mlflow
import pyspark

print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"MLflow  : {mlflow.__version__}")

try:
    import pyspark.ml  # noqa: F401
    import mlflow.spark  # noqa: F401

    print("\nSpark ML available — environment version 5 confirmed")
except ImportError as e:
    raise RuntimeError(
        "Spark ML unavailable. Open the Environment side panel and set environment version 5."
    ) from e


Python  : 3.12.3
PySpark : 4.1.0
MLflow  : 3.8.1

Spark ML available — environment version 4 confirmed


In [0]:
# Namespace DDL — safe to re-run
CATALOG = "workspace"
SCHEMA = "flights"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.artifacts")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Namespace ready: {CATALOG}.{SCHEMA}")
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA}"))


{"ts": "2026-09-13 20:16:03.920", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ1MzkwODE3NzcwMzUwMhABIAEyJDAxYTA5YzY5LTdmYjQtN2Y2OC05ODNlLWMxZjE3ZjczNGQ4NDokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwI+oic1QYQgKKw/wJQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-13 20:16:03.920", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ1MzkwODE3NzcwMzUwMhABIAEyJDAxYTA5YzY5LTdmYjQtN2Y2OC05ODNlLWMxZjE3ZjczNGQ4NDokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwI+oic1QYQgKKw/wJQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-13 20:16:03.920", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ1MzkwODE3NzcwMzUwMhABIAEyJDAxYTA5YzY5LTdmYjQtN2Y2OC05ODNlLWMxZjE3ZjczNGQ4NDokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5

Namespace ready: workspace.flights


database,volume_name
flights,artifacts
flights,raw


In [0]:
# Preflight — confirm the source CSV landed in the volume
RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
SOURCE_FILE = "flights_sample_3m.csv"

files = {f.name: f.size for f in dbutils.fs.ls(RAW_VOLUME)}
if SOURCE_FILE not in files:
    raise FileNotFoundError(
        f"{SOURCE_FILE} not found in {RAW_VOLUME}.\n"
        f"Found: {sorted(files) or '(empty)'}\n\n"
        "Upload it via Catalog Explorer > workspace > flights > raw > Upload to this volume."
    )

size_mb = files[SOURCE_FILE] / 1024**2
print(f"{SOURCE_FILE}  ({size_mb:,.1f} MB)")

# Peek at the header without loading the file
header = spark.read.csv(f"{RAW_VOLUME}/{SOURCE_FILE}", header=True).limit(5)
print(f"Columns: {len(header.columns)}")
display(header)


flights_sample_3m.csv  (585.7 MB)
Columns: 32


FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,null,0.0,186.0,176.0,153.0,1065.0,null,null,null,null,null
2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,null,0.0,235.0,236.0,189.0,1399.0,null,null,null,null,null
2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,null,0.0,118.0,112.0,87.0,680.0,null,null,null,null,null
2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,null,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,null,0.0,181.0,182.0,153.0,985.0,null,null,null,null,null


In [ ]:
# API preflight — every integration checked the same way: resolve the secret,
# make one real call, report the answer and the remaining quota.
#
# Checking that a secret exists proves only that a string is stored. These call
# the APIs, because the failures that matter are the ones a stored string cannot
# reveal: a revoked key, an exhausted quota, a plan that does not cover the
# endpoint. None of the checks is fatal — everything through 05_train runs with
# no API access at all.
from src import config

results = []


def probe(name, fn):
    """Run one live check and record a single-line verdict."""
    try:
        results.append((name, "OK", fn()))
    except Exception as e:
        results.append((name, "FAIL", f"{type(e).__name__}: {e}"))


def _aerodatabox():
    from datetime import date
    from src.aerodatabox import AeroDataBoxClient
    client = AeroDataBoxClient(
        dbutils.secrets.get(config.AERODATABOX_SECRET_SCOPE, config.AERODATABOX_SECRET_KEY))
    legs = client.flight_by_number("DL1572", date.today())
    q = client.last_quota
    return (f"{len(legs)} leg(s) for DL1572 today | "
            f"{q.get('units_remaining')}/{q.get('units_limit')} units, "
            f"{q.get('requests_remaining')}/{q.get('requests_limit')} requests left")


def _opensky():
    from src.opensky import CONUS_BBOX, OpenSkyClient, parse_states, split_by_phase
    client = OpenSkyClient(
        dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_ID_KEY),
        dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_SECRET_KEY))
    rows = parse_states(client.fetch_states(CONUS_BBOX))
    ground, airborne = split_by_phase(rows)
    return f"{len(rows):,} aircraft ({len(ground):,} on ground, {len(airborne):,} airborne)"


def _aviationstack():
    # Retained for the recorded fixture and the unit tests only. The plan serves
    # schedules roughly two weeks old, which is why AeroDataBox took over the
    # live path: a flight from a fortnight ago cannot appear in a live snapshot.
    key = dbutils.secrets.get(
        config.AVIATIONSTACK_SECRET_SCOPE, config.AVIATIONSTACK_SECRET_KEY)
    return "secret present (legacy — not used on the live path)" if key else "empty"


probe("AeroDataBox  schedules + gate times", _aerodatabox)
probe("OpenSky      live state + phase", _opensky)
probe("AviationStack (legacy)", _aviationstack)

width = max(len(n) for n, _, _ in results)
for name, status, detail in results:
    print(f"  {status:<5} {name:<{width}}  {detail}")

failed = [n for n, s, _ in results if s == "FAIL"]
if failed:
    print(f"\n{len(failed)} integration(s) unavailable: {failed}")
    print("Only 06_api_ingest and 07_score need these. 01_bronze through 05_train")
    print("run entirely on the BTS extract with no network access.")
    print("\nTo create a missing secret:")
    print(f"    databricks secrets create-scope {config.AERODATABOX_SECRET_SCOPE}")
    print(f"    databricks secrets put-secret {config.AERODATABOX_SECRET_SCOPE} <key-name>")
else:
    print("\nAll integrations reachable.")


In [0]:
# Summary — paste this output into the RUNBOOK reply block
print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Catalog / schema : {CATALOG}.{SCHEMA}")
print("  Volumes          : raw, artifacts")
print(f"  Source file      : {SOURCE_FILE} ({size_mb:,.1f} MB)")
print(f"  Spark            : {pyspark.__version__}")
print(f"  MLflow           : {mlflow.__version__}")
print("\nNext: notebooks/01_bronze.ipynb")


SETUP COMPLETE
  Catalog / schema : workspace.flights
  Volumes          : raw, artifacts
  Source file      : flights_sample_3m.csv (585.7 MB)
  Spark            : 4.1.0
  MLflow           : 3.8.1

Next: notebooks/01_bronze.ipynb
